# Bronze CDC DDL (Avro Schema)

## Import Helper Functions

In [1]:
from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config_loader import load_config
# from src.spark_sql_magic import sql
# from src.avro_utils import fetch_avro_schema

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
TOPIC_PREFIX = cfg["kafka"]["topic_prefix"]
TOPIC_SUFFIX = set(cfg["kafka"]["topic_suffix"])
APICURIO_URL = cfg["api"]["apicurio_url"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["bronze_cdc_ddl"])
        .getOrCreate()
)

## Function: Create Bronze Table

In [4]:
from pyspark.sql.avro.functions import from_avro

def create_bronze_table(topic: str) -> str:
    table_name = f"{CATALOG}.{BRONZE_NAMESPACE}.{topic}"
    
    if spark.catalog.tableExists(table_name):
        print(f"Table {table_name} exists. Skipping.")
        return

    print(f"Creating Bronze table {table_name}")

    # 1. Fetch Schema
    schema_dict = fetch_avro_schema(topic, APICURIO_URL, TOPIC_PREFIX)
    schema_json = json.dumps(schema_dict)

    # 2. Create empty Kafka DF and parse avro
    df_schema = (
        spark.createDataFrame([], "value BINARY")
        .select(from_avro("value", schema_json).alias("data"))
        .select("data.*")
    )

    # 3. Add Columns (kafka and ingestion metadata)
    df_final = (
        df_schema
            .withColumn("kafka_offset", F.lit(None).cast("long"))
            .withColumn("kafka_partition", F.lit(None).cast("int"))
            .withColumn("kafka_ingest_ts", F.lit(None).cast("timestamp"))
            .withColumn("batch_id", F.lit(None).cast("string"))
            .withColumn("spark_ingest_ts", F.lit(None).cast("timestamp"))
    )

    # 4. Create table
    (
        df_final
        .writeTo(table_name)
        .tableProperty("format-version", "3")                       # important for CDC (iceberg v2+)
        .partitionedBy(F.partitioning.days("spark_ingest_ts"))      # iceberg hidden partitioning (spark 4.0)
        .create()
    )

    print(f"Created {table_name}")

## Run: Create Bronze Table

In [5]:
for topic in TOPIC_SUFFIX:
    create_bronze_table(topic)

Creating Bronze table polaris.bronze.order_items


Created polaris.bronze.order_items
Creating Bronze table polaris.bronze.customers
Created polaris.bronze.customers
Creating Bronze table polaris.bronze.order_payments
Created polaris.bronze.order_payments
Creating Bronze table polaris.bronze.orders
Created polaris.bronze.orders
Creating Bronze table polaris.bronze.products
Created polaris.bronze.products
Creating Bronze table polaris.bronze.sellers
Created polaris.bronze.sellers
Creating Bronze table polaris.bronze.order_reviews
Created polaris.bronze.order_reviews


## Sanity Check All Tables

In [6]:
%%sql
SHOW TABLES IN bronze;

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|bronze   |order_payments|false      |
|bronze   |orders        |false      |
|bronze   |sellers       |false      |
|bronze   |order_reviews |false      |
|bronze   |order_items   |false      |
|bronze   |products      |false      |
|bronze   |customers     |false      |
+---------+--------------+-----------+



In [7]:
for item in TOPIC_SUFFIX:
    df = spark.table(f"polaris.bronze.{item}")
    df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: long (nullable = true)
 |-- price: struct (nullable = true)
 |    |-- scale: integer (nullable = true)
 |    |-- value: binary (nullable = true)
 |-- freight_value: struct (nullable = true)
 |    |-- scale: integer (nullable = true)
 |    |-- value: binary (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)
 |-- __op: string (nullable = true)
 |-- __ts_ms: long (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_ingest_ts: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- spark_ingest_ts: timestamp (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (null

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  